This notebook orchestrates gold-layer BI reporting view creation by running one focused notebook per view under `/Users/sharif.sk@zohomail.in/adwm-adb-rep/gold/bi`.

Execution order:

* `./FactSales BI Base View`
* `./FactSales BI Monthly Trend View`
* `./FactSales BI Territory View`
* `./FactSales BI Category View`
* `./FactSales BI Salesperson View`

Scope:

* Keep each BI view in its own notebook
* Provide a single entry point for sequential execution
* Retain shared validations after all child notebooks finish

Notes:

* The base view notebook should run first because the aggregate BI views depend on `adwm_wh.gold.vw_bi_factsales_base`
* Category outputs may still surface `Unknown` values until upstream product dimension enrichment is resolved

Child notebooks created under the BI folder:

* [FactSales BI Base View](#notebook-1309605259360158)
* [FactSales BI Monthly Trend View](#notebook-1309605259360159)
* [FactSales BI Territory View](#notebook-1309605259360160)
* [FactSales BI Category View](#notebook-1309605259360161)
* [FactSales BI Salesperson View](#notebook-1309605259360162)

Run this notebook top to bottom to recreate all BI views and then execute the shared validations below.

In [0]:
%run "./FactSales BI Base View"

Runs the monthly trend child notebook after the base BI view is refreshed.

In [0]:
%run "./FactSales BI Monthly Trend View"

Runs the territory aggregate child notebook after the base BI view is refreshed.

In [0]:
%run "./FactSales BI Territory View"

Runs the category aggregate child notebook. Current outputs may still group under `Unknown` until upstream product attributes are enriched.

In [0]:
%run "./FactSales BI Category View"

Runs the salesperson aggregate child notebook after the base BI view is refreshed.

In [0]:
%run "./FactSales BI Salesperson View"

Shared validations below confirm that the split BI notebooks produced the expected reusable views.

Checks:

* Row counts and available date range from the base and monthly views
* Presence of `Unknown` dimension buckets in the base view
* Top rows from each aggregate view for a quick sanity check

In [0]:
%sql
SELECT 'vw_bi_factsales_base' AS view_name,
       COUNT(*) AS row_count,
       MIN(FullDate) AS min_full_date,
       MAX(FullDate) AS max_full_date,
       CAST(SUM(SalesAmount) AS DECIMAL(19,4)) AS total_sales_amount
FROM adwm_wh.gold.vw_bi_factsales_base
UNION ALL
SELECT 'vw_bi_monthly_sales_trend' AS view_name,
       COUNT(*) AS row_count,
       MIN(PeriodStartDate) AS min_full_date,
       MAX(PeriodEndDate) AS max_full_date,
       CAST(SUM(TotalSalesAmount) AS DECIMAL(19,4)) AS total_sales_amount
FROM adwm_wh.gold.vw_bi_monthly_sales_trend
UNION ALL
SELECT 'vw_bi_sales_by_territory' AS view_name,
       COUNT(*) AS row_count,
       CAST(NULL AS DATE) AS min_full_date,
       CAST(NULL AS DATE) AS max_full_date,
       CAST(SUM(TotalSalesAmount) AS DECIMAL(19,4)) AS total_sales_amount
FROM adwm_wh.gold.vw_bi_sales_by_territory
UNION ALL
SELECT 'vw_bi_sales_by_category' AS view_name,
       COUNT(*) AS row_count,
       CAST(NULL AS DATE) AS min_full_date,
       CAST(NULL AS DATE) AS max_full_date,
       CAST(SUM(TotalSalesAmount) AS DECIMAL(19,4)) AS total_sales_amount
FROM adwm_wh.gold.vw_bi_sales_by_category
UNION ALL
SELECT 'vw_bi_sales_by_salesperson' AS view_name,
       COUNT(*) AS row_count,
       CAST(NULL AS DATE) AS min_full_date,
       CAST(NULL AS DATE) AS max_full_date,
       CAST(SUM(TotalSalesAmount) AS DECIMAL(19,4)) AS total_sales_amount
FROM adwm_wh.gold.vw_bi_sales_by_salesperson
ORDER BY view_name;

In [0]:
%sql
SELECT
  SUM(CASE WHEN TerritoryName = 'Unknown' THEN 1 ELSE 0 END) AS unknown_territory_rows,
  SUM(CASE WHEN CountryRegion = 'Unknown' THEN 1 ELSE 0 END) AS unknown_country_rows,
  SUM(CASE WHEN CategoryName = 'Unknown' THEN 1 ELSE 0 END) AS unknown_category_rows,
  SUM(CASE WHEN SubcategoryName = 'Unknown' THEN 1 ELSE 0 END) AS unknown_subcategory_rows,
  SUM(CASE WHEN EmployeeFullName = 'Unknown' THEN 1 ELSE 0 END) AS unknown_employee_rows,
  COUNT(*) AS total_rows
FROM adwm_wh.gold.vw_bi_factsales_base;

In [0]:
%sql
SELECT report_name,
       label_1,
       label_2,
       CAST(total_sales_amount AS DECIMAL(19,2)) AS total_sales_amount,
       total_order_quantity,
       CAST(gross_margin AS DECIMAL(19,2)) AS gross_margin,
       CAST(total_discount_amount AS DECIMAL(19,2)) AS total_discount_amount
FROM (
  SELECT
    'monthly_sales_trend' AS report_name,
    YearMonth AS label_1,
    MonthName AS label_2,
    TotalSalesAmount AS total_sales_amount,
    TotalOrderQuantity AS total_order_quantity,
    GrossMargin AS gross_margin,
    TotalDiscountAmount AS total_discount_amount,
    ROW_NUMBER() OVER (PARTITION BY 'monthly_sales_trend' ORDER BY TotalSalesAmount DESC, YearMonth) AS rn
  FROM adwm_wh.gold.vw_bi_monthly_sales_trend

  UNION ALL

  SELECT
    'sales_by_territory' AS report_name,
    TerritoryName AS label_1,
    CountryRegion AS label_2,
    TotalSalesAmount AS total_sales_amount,
    TotalOrderQuantity AS total_order_quantity,
    GrossMargin AS gross_margin,
    TotalDiscountAmount AS total_discount_amount,
    ROW_NUMBER() OVER (PARTITION BY 'sales_by_territory' ORDER BY TotalSalesAmount DESC, TerritoryName, CountryRegion) AS rn
  FROM adwm_wh.gold.vw_bi_sales_by_territory

  UNION ALL

  SELECT
    'sales_by_category' AS report_name,
    CategoryName AS label_1,
    SubcategoryName AS label_2,
    TotalSalesAmount AS total_sales_amount,
    TotalOrderQuantity AS total_order_quantity,
    GrossMargin AS gross_margin,
    TotalDiscountAmount AS total_discount_amount,
    ROW_NUMBER() OVER (PARTITION BY 'sales_by_category' ORDER BY TotalSalesAmount DESC, CategoryName, SubcategoryName) AS rn
  FROM adwm_wh.gold.vw_bi_sales_by_category

  UNION ALL

  SELECT
    'sales_by_salesperson' AS report_name,
    EmployeeFullName AS label_1,
    DepartmentName AS label_2,
    TotalSalesAmount AS total_sales_amount,
    TotalOrderQuantity AS total_order_quantity,
    GrossMargin AS gross_margin,
    TotalDiscountAmount AS total_discount_amount,
    ROW_NUMBER() OVER (PARTITION BY 'sales_by_salesperson' ORDER BY TotalSalesAmount DESC, EmployeeFullName, DepartmentName) AS rn
  FROM adwm_wh.gold.vw_bi_sales_by_salesperson
) ranked
WHERE rn <= 5
ORDER BY report_name, total_sales_amount DESC, label_1;